In [1]:
import base64
import re

import pandas as pd
from zhconv import convert


def get_tokens_series():
    tokenizer = pd.read_csv("data/o200k_base.tiktoken", header=None)[0]

    def decode_base64(entry):
        base64_part = entry.split(" ")[0]
        decoded_bytes = base64.b64decode(base64_part)

        for encode in ["utf-8", "latin-1"]:
            try:
                return decoded_bytes.decode(encode)
            except UnicodeDecodeError:
                continue
        return ""

    tokens = tokenizer.apply(decode_base64)
    return tokens

In [2]:
def get_chars_dict():
    zh_chars = set()
    jp_chars = set()
    with open("data/Unihan_IRGSources.txt", "r") as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            fields = line.split()
            zh_sources = ["G", "T", "H"]
            jp_sources = ["J"]
            if fields[2].startswith(tuple(zh_sources)):
                code_point = int(fields[0][2:], 16)
                char = chr(code_point)
                zh_chars.add(char)
            if fields[2].startswith(tuple(jp_sources)):
                code_point = int(fields[0][2:], 16)
                char = chr(code_point)
                jp_chars.add(char)
    return zh_chars, jp_chars

In [3]:
def get_words_dict():
    with open("data/dict.txt.small", "r") as f:
        words = f.read().splitlines()

    jieba_dict = [w.split(" ")[0] for w in words]

    zh_dict = {convert(w, "zh-cn") for w in jieba_dict}.union(
        {convert(w, "zh-tw") for w in jieba_dict}
    )
    return zh_dict

In [4]:
zh_chars, jp_chars = get_chars_dict()
zh_dict = get_words_dict()

o200k_tokens = get_tokens_series()
print("Total tokens:", len(o200k_tokens))

Total tokens: 199998


### Filter Chinese Tokens
Tokens that contains characters from the G, H, and W range of [Unihan](http://www.unicode.org/reports/tr38/#kIRG_TSource) database are most likely Chinese tokens.

In [5]:
filtered = o200k_tokens[o200k_tokens.apply(lambda x: any(c in zh_chars for c in x))]

filtered = filtered[~filtered.str.contains(r"[\u3040-\u309F\u30A0-\u30FF]", regex=True)]

print("Tokens with Chinese characters:", len(filtered))

Tokens with Chinese characters: 7452


###  Identify Strange Tokens

The tokens are further filtered based on:
 - A single character token are considered normal
 - Words that can be found in the jieba dictionary are considered normal
 - All space and punctuations are remove from the token when comparing with the dictionary.

In [6]:
candidates = filtered[
    filtered.apply(lambda x: re.sub(r"[^\w]", "", x) not in zh_dict)
    & filtered.apply(lambda x: len(x) >= 2)
]

print("Strange tokens candidates:", len(candidates))

Strange tokens candidates: 1713


### Semi-mannually labelling 

In [7]:
df = candidates.sort_values(key=lambda x: x.str.len(), ascending=False).reset_index()
df.columns = ["original index in o200k", "token"]
df["category"] = ""
df

,original index in o200k,token,category
0,185118,_日本毛片免费视频观看,
1,181081,微信公众号天天中彩票,
2,187716,微信里的天天中彩票,
3,188394,天天中彩票大神推荐,
4,170996,微信上的天天中彩票,
...,...,...,...
1708,109696,一覧,
1709,109720,是谁,
1710,109810,日起,
1711,110755,自产,


In [8]:
df.to_csv("results/candidates.csv", index=False)

**Find the most frequency bigram across the series**

In [9]:
from collections import defaultdict


def count_bigram_presence(series):
    bigram_presence = defaultdict(int)

    for entry in series:
        bigrams = set(entry[i : i + 2] for i in range(len(entry) - 1))
        for bigram in bigrams:
            bigram_presence[bigram] += 1

    return bigram_presence


bigram_presence_count = count_bigram_presence(df["token"])

bigram_presence_df = pd.DataFrame(
    list(bigram_presence_count.items()), columns=["Bigram", "Frequency"]
)
bigram_presence_df.sort_values("Frequency", ascending=False, inplace=True)

bigram_presence_df[0:10]

,Bigram,Frequency
14,彩票,205
12,天天,155
27,天,106
10,天中,90
18,中彩,89
33,大发,60
110,在线,47
36,大,42
149,争霸,39
8,免费,39


**Step1: identify "彩票" related**

In [10]:
print(df[df["token"].str.contains("彩票")]["token"].to_list())

[' 微信公众号天天中彩票', ' 微信里的天天中彩票', ' 天天中彩票大神推荐', ' 微信上的天天中彩票', ' 天天中彩票为什么', ' 天天彩票与你同行', ' 微信的天天中彩票', ' 天天中彩票不中返', ' 天天中彩票一等奖', ' 天天中彩票公众号', ' 天天中彩票中大奖', ' 天天中彩票不能买', ' 天天中彩票中奖了', ' 天天中彩票APP', ' 天天中彩票nba', ' 手机上天天中彩票', ' 手机版天天中彩票', ' 天天中彩票是不是', ' qq的天天中彩票', ' 天天中彩票双色球', ' 中国福利彩票天天', ' 天天中彩票怎么买', ' 天天爱彩票app', ' 全民彩票天天送钱', ' 天天中彩票app', ' 天天中彩票提现', ' 天天爱彩票网站', ' 天天中彩票微信', ' 天天中彩票有人', ' 天天中彩票这个', ' 天天中彩票投注', ' 天天中彩票人工', ' 天天中彩票彩金', ' 天天中彩票大奖', ' 微信天天中彩票', ' 福利彩票天天彩', ' 天天彩票app', ' 天天中彩票任选', ' 天天中彩票公司', ' 天天中彩票篮球', ' 天天中彩票腾讯', '公众号天天中彩票', ' 天天中彩票怎样', ' 天天彩票中大奖', ' 天天中彩票网络', ' 天天爱彩票提现', ' 天天中彩票网站', ' 天天中彩票在哪', ' 天天中彩票中奖', ' 天天中彩票派奖', ' 天天爱彩票怎么', ' 天天中彩票实名', ' 天天中彩票出票', ' 天天中彩票软件', ' 天天大奖彩票站', ' 腾讯天天中彩票', ' 全民彩票天天送', ' 天天中彩票如何', ' 天天中彩票官方', ' 天天中彩票足彩', ' 海南天天中彩票', ' 天天中彩票开奖', ' 天天中彩票追号', ' 天天中彩票可以', ' 天天中彩票足球', ' 天天中彩票提款', ' 天天中彩票中了', ' 天天中彩票无法', ' 天天爱彩票中奖', ' 手机天天中彩票', ' 天天中彩票充值', ' 天天中彩票qq', ' 天天中彩票官网', ' qq天天中彩票', ' 天天中彩票怎么', ' 天天中彩票不能', ' 天天中彩票的', ' 天天彩票提现', ' 天天彩票中奖', ' 天天送钱彩票', ' 天天中彩票在', ' 

In [11]:
df.loc[df["token"].str.contains("彩票"), "category"] = "lottery / gamble"

In [12]:
print(df[df["token"].str.contains("彩") & df["category"].eq("")]["token"].to_list())

[' 彩神争霸官方下载', ' 彩神争霸大发快三', ' 彩神争霸邀请码', ' 大发时时彩开奖', ' 彩神争霸是不是', ' 彩神争霸怎么样', ' 彩神争霸电脑版', ' 大发时时彩计划', ' 彩神争霸大发快', ' 大发时时彩怎么', ' 高频彩大发快三', ' 彩神争霸app', ' 彩神争霸怎么', ' 彩神争霸安卓', ' 彩神争霸官网', ' 彩神争霸破解', ' 重庆时时彩杀', ' 彩神争霸大发', ' 彩神争霸可以', ' 彩神争霸充值', ' 彩神争霸如何', '彩神争霸邀请码', ' 彩神争霸代理', ' 彩神争霸平台', ' 大发时时彩是', ' 彩神争霸提现', ' 重庆时时彩的', ' 彩神争霸网站', ' 彩神争霸快三', ' 彩神争霸官方', ' 彩神争霸下载', ' 重庆时时彩彩', ' 彩神争霸苹果', ' 彩神争霸输钱', ' 大发时时彩', ' 彩神争霸有', ' 彩神争霸快', ' 彩神争霸能', ' 香港六合彩', ' 大发分分彩', '彩网大发快三', '下载彩神争霸', ' 彩神争霸高', ' 时时彩平台', ' 彩神争霸是', ' 腾讯分分彩', ' 重庆时时彩', ' 彩神争霸的', ' 玩彩神争霸', '腾讯分分彩', '玩彩神争霸', ' 时时彩后', '时时彩开奖', '彩娱乐平台', '时时彩官网', ' 神彩争霸', '时时彩平台', '分分彩计划', '大发时时彩', '彩大发快三', '时时彩计划', '重庆时时彩', ' 彩神争霸', ' 老时时彩', ' 天天彩', '天天好彩', ' 三分彩', '老时时彩', '竞彩足球', ' 七星彩', '购彩平台', '购彩官网', ' 五分彩', '分时时彩', ' 一分彩', ' 六和彩', ' 分分彩', ' 时时彩', '博彩公司', '彩神争霸', '天下彩', '彩注册', ' 体彩', '福彩快', ' 乐彩', '二分彩', ' 杏彩', '彩在线', '彩争霸', ' 彩神', '送彩金', ' 福彩', '七星彩', '三分彩', '五分彩', '时时彩', '一分彩', '分分彩', '彩娱乐', '彩平台', '体彩', '乐彩', '彩网', '彩金', '彩神', '分彩', '足彩', '星彩', '购彩', '福彩'